In [1]:
import pandas as pd
import numpy as np

# loading Cleaned Datasets

In [ ]:
cleaned_data_path = '../data/processed/'


def load_dataset(path):
    df = pd.read_csv(f'{cleaned_data_path}{path}')
    return df

# Load all raw CSV files
account_profiles_df_processed = load_dataset("account_profiles_clean.csv")
fraud_patterns_df_processed = load_dataset("fraud_patterns_clean.csv")
network_edges_df_processed = load_dataset("network_edges_clean.csv")
time_series_stats_df_processed = load_dataset("time_series_stats_clean.csv")
transactions_df_processed = load_dataset("transactions_clean.csv")

print("All cleaned CSV files loaded.")

# 1. Test Final Merged Datasets

In [ ]:
print("### Merging Transactions with Account Profiles...")

# Merge transactions with account profiles to enrich transaction data
# We'll merge on 'account_id'
merged_transactions_df = pd.merge(
    transactions_df_processed,
    account_profiles_df_processed,
    on='account_id',
    how='left',
    suffixes=('_txn', '_acc')
)

print("Merged DataFrame Shape:", merged_transactions_df.shape)
print("Merged DataFrame Columns:", merged_transactions_df.columns.tolist()[:10], "...") # Display first 10 columns
display(merged_transactions_df.head())
print("\nInfo of Merged DataFrame:")
merged_transactions_df.info()

# 2. Dashboard KPI Tables

In [ ]:
print("### Creating Overall Fraud Metrics KPI Table")

overall_fraud_kpis = pd.DataFrame({
    'Metric': [
        'Total Transactions',
        'Total Fraudulent Transactions',
        'Fraud Rate (%)',
        'Total Amount Transacted',
        'Total Fraud Amount',
        'Average Transaction Amount',
        'Average Fraud Amount'
    ],
    'Value': [
        len(merged_transactions_df),
        merged_transactions_df['is_fraud'].sum(),
        merged_transactions_df['is_fraud'].mean() * 100,
        merged_transactions_df['amount'].sum(),
        merged_transactions_df.loc[merged_transactions_df['is_fraud'] == True, 'amount'].sum(),
        merged_transactions_df['amount'].mean(),
        merged_transactions_df.loc[merged_transactions_df['is_fraud'] == True, 'amount'].mean()
    ]
})

display(overall_fraud_kpis)

print("\n### Creating Fraud Metrics by Merchant Category")
merchant_fraud_kpis = merged_transactions_df.groupby('merchant_category').agg(
    total_transactions=('transaction_id', 'count'),
    total_fraud_transactions=('is_fraud', 'sum'),
    total_amount=('amount', 'sum'),
    total_fraud_amount=('amount', lambda x: x[merged_transactions_df.loc[x.index, 'is_fraud']].sum())
).reset_index()
merchant_fraud_kpis['fraud_rate_pct'] = (merchant_fraud_kpis['total_fraud_transactions'] / merchant_fraud_kpis['total_transactions']) * 100
merchant_fraud_kpis = merchant_fraud_kpis.sort_values('fraud_rate_pct', ascending=False)
display(merchant_fraud_kpis.head())

print("\n### Creating Fraud Metrics by Account Type")
account_type_fraud_kpis = merged_transactions_df.groupby('account_type').agg(
    total_transactions=('transaction_id', 'count'),
    total_fraud_transactions=('is_fraud', 'sum'),
    total_amount=('amount', 'sum'),
    total_fraud_amount=('amount', lambda x: x[merged_transactions_df.loc[x.index, 'is_fraud']].sum())
).reset_index()
account_type_fraud_kpis['fraud_rate_pct'] = (account_type_fraud_kpis['total_fraud_transactions'] / account_type_fraud_kpis['total_transactions']) * 100
account_type_fraud_kpis = account_type_fraud_kpis.sort_values('fraud_rate_pct', ascending=False)
display(account_type_fraud_kpis.head())